In [1]:
import os
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Establish connection with SQLite DB
DB_PATH = os.path.join("..", "data", "olist.db")
conn = sqlite3.connect(DB_PATH)

# 2. Query of data extraction with JOINs and basic transformation
query = """
SELECT 
    o.order_id,
    o.customer_id,
    c.customer_state,
    c.customer_city,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    i.price,
    i.freight_value,
    p.payment_type,
    p.payment_value,
    r.review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN (
    SELECT order_id, SUM(price) as price, SUM(freight_value) as freight_value 
    FROM order_items 
    GROUP BY order_id
) i ON o.order_id = i.order_id
LEFT JOIN (
    SELECT order_id, payment_type, SUM(payment_value) as payment_value 
    FROM payments 
    GROUP BY order_id
) p ON o.order_id = p.order_id
LEFT JOIN (
    SELECT order_id, AVG(review_score) as review_score 
    FROM reviews 
    GROUP BY order_id
) r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered';
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f"Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded: 96,478 rows and 13 columns.


,order_id,customer_id,customer_state,customer_city,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,payment_type,payment_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,SP,sao paulo,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18 00:00:00,29.99,8.72,credit_card,38.71,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,BA,barreiras,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13 00:00:00,118.70,22.76,boleto,141.46,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,GO,vianopolis,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04 00:00:00,159.90,19.22,credit_card,179.12,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,RN,sao goncalo do amarante,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15 00:00:00,45.00,27.20,credit_card,72.20,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,SP,santo andre,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26 00:00:00,19.90,8.72,credit_card,28.62,5.0


In [2]:
#Convert date columns to datetime
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

# Actual number of days the delivery took
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

# Difference against the estimated time (Days of delay/advance)
# If positive = delayed delivery | If negative = early delivery
df['delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Binary flag: Did it arrive late?
df['is_delayed'] = df['delay_days'] > 0

# Visualize first statistics
print(f"Percentage of orders delivered with delay: {df['is_delayed'].mean() * 100:.2f}%")
df[['delivery_time_days', 'delay_days', 'price', 'freight_value', 'review_score']].describe()

Percentage of orders delivered with delay: 8.11%


,delivery_time_days,delay_days,price,freight_value,review_score
count,96470.000000,96470.000000,96478.000000,96478.000000,95832.000000
mean,12.558217,-11.178126,137.041586,22.785253,4.156187
std,9.546156,10.184354,209.045198,21.559197,1.283622
min,0.533414,-146.016123,0.850000,0.000000,1.000000
25%,6.766204,-16.244065,45.900000,13.850000,4.000000
50%,10.217477,-11.948102,86.575000,17.170000,5.000000
75%,15.720182,-6.389815,149.900000,24.017500,5.000000
max,209.628611,188.975081,13440.000000,1794.960000,5.000000


In [3]:
# Group the average rating by delay status
df['delay_status'] = df['is_delayed'].map({True: 'Late', False: 'On Time / Early'})

fig = px.box(
    df, 
    x='delay_status', 
    y='review_score', 
    color='delay_status',
    title='Impact of Delivery Status on Ratings (Review Score)',
    labels={'review_score': 'Rating (1-5)', 'delay_status': 'Delivery Status'},
    color_discrete_map={'Late': '#EF553B', 'On Time / Early': '#636EFA'}
)

fig.show()

In [4]:
# Calculate averages and exact metrics by delivery status
metrics = df.groupby('delay_status')['review_score'].agg(
    average='mean',
    median='median',
    standard_deviation='std',
    total_orders='count'
).reset_index()

# Calculate the percentage of customers who gave 1 star
df['is_one_star'] = df['review_score'] == 1
one_star_pct = df.groupby('delay_status')['is_one_star'].mean() * 100

metrics['percentage_one_star'] = one_star_pct.values
print(metrics.to_string(index=False))

   delay_status  average  median  standard_deviation  total_orders  percentage_one_star
           Late 2.566506     2.0            1.657815          7661            45.144390
On Time / Early 4.294311     5.0            1.146048         88171             6.519875


In [7]:
query_ml = """
SELECT 
    o.order_id,
    o.customer_id,
    c.customer_state,
    s.seller_state,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,
    i.price,
    i.freight_value,
    o.order_purchase_timestamp,
    o.order_estimated_delivery_date,
    o.order_delivered_customer_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items i ON o.order_id = i.order_id
JOIN products p ON i.product_id = p.product_id
JOIN sellers s ON i.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL;
"""

#Establish connection with SQLite DB and load data for ML
conn = sqlite3.connect(DB_PATH)
df_ml = pd.read_sql_query(query_ml, conn)

# Delete duplicated rows if an order has multiple items
df_ml = df_ml.drop_duplicates(subset=['order_id']).copy()

# Parse dates
df_ml['order_purchase_timestamp'] = pd.to_datetime(df_ml['order_purchase_timestamp'])
df_ml['order_estimated_delivery_date'] = pd.to_datetime(df_ml['order_estimated_delivery_date'])
df_ml['order_delivered_customer_date'] = pd.to_datetime(df_ml['order_delivered_customer_date'])

# Objetive variable (Target): 1 if delayed, 0 if not
df_ml['target_delayed'] = (df_ml['order_delivered_customer_date'] > df_ml['order_estimated_delivery_date']).astype(int)

# 1. Feature: Delivery State (1) vs Interstate (0)
df_ml['is_same_state'] = (df_ml['customer_state'] == df_ml['seller_state']).astype(int)

# 2. Feature: Estimated Days of Delivery Promised by the Platform
df_ml['estimated_days_promised'] = (df_ml['order_estimated_delivery_date'] - df_ml['order_purchase_timestamp']).dt.total_seconds() / 86400

# 3. Feature: Proportion of freight cost over total price
df_ml['freight_ratio'] = df_ml['freight_value'] / (df_ml['price'] + df_ml['freight_value'] + 1e-5)

# 4. Feature: Approximate volume of the package in cm³
df_ml['product_volume_cm3'] = df_ml['product_length_cm'] * df_ml['product_height_cm'] * df_ml['product_width_cm']

# 5. Features temporales: Day of the week and hour of purchase
df_ml['purchase_dayofweek'] = df_ml['order_purchase_timestamp'].dt.dayofweek
df_ml['purchase_hour'] = df_ml['order_purchase_timestamp'].dt.hour

# clean rows with null values in dimensions or weights
features = [
    'price', 'freight_value', 'freight_ratio', 'is_same_state', 
    'estimated_days_promised', 'product_weight_g', 'product_volume_cm3',
    'purchase_dayofweek', 'purchase_hour'
]

df_ml_clean = df_ml.dropna(subset=features + ['target_delayed']).copy()

print(f"Dataset ready for ML: {df_ml_clean.shape[0]:,} registros.")
print(f"Distribution of the target (0 = On Time, 1 = Delayed):\n{df_ml_clean['target_delayed'].value_counts(normalize=True) * 100}")
conn.close()
df_ml_clean = df_ml.dropna(subset=features + ['target_delayed']).copy()
df_ml_clean.head()


Dataset ready for ML: 96,454 registros.
Distribution of the target (0 = On Time, 1 = Delayed):
target_delayed
0    91.886288
1     8.113712
Name: proportion, dtype: float64


C:\Users\Dell\AppData\Local\Temp\ipykernel_8984\2093707617.py:54: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_ml['purchase_dayofweek'] = df_ml['order_purchase_timestamp'].dt.dayofweek


,order_id,customer_id,customer_state,seller_state,product_weight_g,product_length_cm,product_height_cm,product_width_cm,price,freight_value,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,target_delayed,is_same_state,estimated_days_promised,freight_ratio,product_volume_cm3,purchase_dayofweek,purchase_hour
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,SP,SP,500.0,19.0,8.0,13.0,29.99,8.72,2017-10-02 10:56:33,2017-10-18,2017-10-10 21:25:13,0,1,15.544063,0.225265,1976.0,0,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,BA,SP,400.0,19.0,13.0,19.0,118.70,22.76,2018-07-24 20:41:37,2018-08-13,2018-08-07 15:27:45,0,0,19.137766,0.160894,4693.0,1,20
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,GO,SP,420.0,24.0,19.0,21.0,159.90,19.22,2018-08-08 08:38:49,2018-09-04,2018-08-17 18:06:29,0,0,26.639711,0.107302,9576.0,2,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,RN,MG,450.0,30.0,10.0,20.0,45.00,27.20,2017-11-18 19:28:06,2017-12-15,2017-12-02 00:28:42,0,0,26.188819,0.376731,6000.0,5,19
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,SP,SP,250.0,51.0,15.0,15.0,19.90,8.72,2018-02-13 21:18:39,2018-02-26,2018-02-16 18:17:02,0,1,12.112049,0.304682,11475.0,1,21


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = df_ml_clean[features]
y = df_ml_clean['target_delayed']

# Split 80% train / 20% test 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Handle the class imbalance with class_weight='balanced'
model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=12, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)
#Training the model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Evaluation Metrics
print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# Feature Importance
importances = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFEATURE IMPORTANCE")
print(importances.to_string(index=False))

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.94      0.69      0.80     17726
           1       0.13      0.52      0.21      1565

    accuracy                           0.68     19291
   macro avg       0.54      0.61      0.50     19291
weighted avg       0.88      0.68      0.75     19291

ROC-AUC Score: 0.6670

FEATURE IMPORTANCE
                feature  importance
estimated_days_promised    0.328113
          freight_value    0.147899
     product_volume_cm3    0.104261
                  price    0.095194
          freight_ratio    0.094179
       product_weight_g    0.088941
          purchase_hour    0.056387
          is_same_state    0.048798
     purchase_dayofweek    0.036229


In [11]:
import joblib
import os

# Save model
model_path = os.path.join("..", "data", "model_rf.joblib")
joblib.dump(model, model_path)
print(f"Model exported successfully to: {model_path}")

Model exported successfully to: ..\data\model_rf.joblib
